# Choosing the right chart

_Data Wrangling_

**Pick the chart from the question you are answering and the data type — not from habit.**

A chart is an answer to a question, drawn with position and length that the eye can
       read. The same dataset can answer several different questions, and each question wants a different
       chart. So you never pick a chart for a dataset — you pick a chart for a (question, data type)
       pair.

       Map your question to one of five intents — comparison, distribution, relationship, composition,
       trend — then look at the types of the variables involved (categorical, numeric, time, geo) and
       how many there are. That pair lands you on exactly one chart family. The skill is mechanical once
       you name the intent and the types honestly.

---

This notebook draws one chart per intent on a single dataset, then contrasts a right vs wrong chart on real data. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — matplotlib + seaborn

### Step 1 — Load data and set up a 2×2 grid

We use the `tips` dataset — one row per restaurant bill — and create a 2×2 figure. The same dataset can answer four different questions, so we draw one chart per panel and let each question pick its own chart family.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

tips = sns.load_dataset("tips")   # bundled: one row per restaurant bill
fig, ax = plt.subplots(2, 2, figsize=(11, 8))

### Step 2 — Distribution and comparison

The top row covers two intents. **Distribution** (how is one numeric column spread out?) wants a histogram. **Comparison** (which category is bigger?) wants a bar chart — and sorting the bars makes the ranking instantly readable.

In [ ]:
# DISTRIBUTION: how is one numeric column spread out? -> histogram.
sns.histplot(tips, x="total_bill", bins=20, ax=ax[0, 0])
ax[0, 0].set_title("Distribution -> histogram of total_bill")

# COMPARISON: which category is bigger? -> bar chart (sorted).
by_day = tips.groupby("day", observed=True)["total_bill"].mean().sort_values()
sns.barplot(x=by_day.values, y=by_day.index, orient="h", ax=ax[0, 1])
ax[0, 1].set_title("Comparison -> bar of mean bill per day")

### Step 3 — Relationship and trend

The bottom row covers two more intents. **Relationship** (do two numbers move together?) wants a scatter. **Trend** (how does a number change along an ordered axis?) wants a line — here party `size` is the ordered axis, and the connected line asserts that ordering.

In [ ]:
# RELATIONSHIP: do two numbers move together? -> scatter.
sns.scatterplot(tips, x="total_bill", y="tip", ax=ax[1, 0])
ax[1, 0].set_title("Relationship -> scatter of tip vs total_bill")

# TREND: how does a number change along an ordered axis? -> line.
# party size is an ordered axis here; mean tip rises with it.
by_size = tips.groupby("size", observed=True)["tip"].mean()
ax[1, 1].plot(by_size.index, by_size.values, marker="o")
ax[1, 1].set_title("Trend -> line of mean tip vs party size")
ax[1, 1].set_xlabel("size")
ax[1, 1].set_ylabel("tip")

### Step 4 — Render the four panels

With all four axes filled, tighten the layout and show the figure. One dataset, four intents, four charts — each matched to its question and data type.

In [ ]:
plt.tight_layout()
plt.show()
# Same dataset, four intents, four charts -- each chart matches its question + data type.

## Visualize the data & results

_To show how the 'mean radius' of breast-cancer cells is spread out, do you draw the distribution (right) or a single mean bar (wrong)? The bar hides the spread the question asks about._

### Step 1 — Load one numeric column and subsample

We pull the real `mean radius` measurement from 569 breast-cancer cell nuclei, then subsample 60 values (seeded) so the example is small and reproducible. The question we want to answer is how this one numeric column is *spread out* — a distribution question.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer

d = load_breast_cancer()                         # 569 real cell-nucleus measurements
feature_index = list(d.feature_names).index('mean radius')
radius = d.data[:, feature_index]

rng = np.random.RandomState(0)
sample_idx = rng.choice(len(radius), 60, replace=False)
radius = radius[sample_idx]                      # subsample to 60

### Step 2 — The wrong chart: a single mean bar

A bar chart shows one number. Collapsing the whole column to its mean answers a *comparison* question, not a distribution one — it throws away the width, skew, and outliers that the question is actually about.

In [ ]:
# WRONG chart: a single mean bar collapses everything to one number.
print('mean radius          :', round(radius.mean(), 1))   # -> 14.1  (hides the spread)

### Step 3 — The right chart: a histogram

A histogram bins the values and counts how many fall in each bin, keeping the full shape — where the data clusters, how wide it is, and where the extremes lie. That is exactly what a distribution question asks for.

In [ ]:
# RIGHT chart: a histogram keeps the full shape.
edges = np.linspace(radius.min(), radius.max(), 8)
counts = np.histogram(radius, bins=edges)[0]
print('histogram bin edges  :', np.round(edges, 1))         # -> [ 7.7  9.7 11.7 13.7 15.7 17.7 19.7 21.8]
print('histogram counts     :', counts)                     # -> [ 5  9 19 12  4  5  6]
print('min / max            :', round(radius.min(), 1), '/', round(radius.max(), 1))   # -> 7.7 / 21.8

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate shows market share for 9 brands as a pie chart. The three biggest slices look nearly the same size and you cannot tell which brand leads. What is wrong, and what chart fixes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name the intent: ranking which brand is biggest is a comparison. — _Comparison wants a chart that encodes value as length on a common baseline._
- Diagnose the pie: 9 slices, several similar in size, encoded as angle and area. — _The eye reads angle and area poorly, so similar slices are indistinguishable — the pitfall of pies with many or similar slices._
- Switch to a bar chart, one bar per brand, sorted from largest to smallest. — _Sorted bars on a shared baseline use position and length, the most accurate channel, so the ranking is obvious._

**Answer:** It is a pie with too many similar slices. The intent is comparison, which wants length on a common baseline, but a pie encodes value as angle and area — the channels the eye reads worst. Replace it with a sorted bar chart (or a dot plot), where the leading brand is the tallest bar.

</details>

**Problem 2.** You want to report a sensor's readings and you draw one bar showing the mean reading. A reviewer says the bar "hides the data". What did the bar miss, and which charts should you use instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name the intent: showing how the readings are spread is a distribution question over one numeric column. — _A distribution is about center, width, skew, and outliers — not a single summary number._
- See why a mean bar fails: it collapses the whole column to one height. — _Two very different spreads (tight vs wide, symmetric vs skewed) can have the identical mean, so the bar hides exactly what you wanted to show._
- Use a histogram (or box plot / violin / ECDF) of the readings. — _These show the full shape — width, skew, and outliers — which is the point of a distribution._

**Answer:** A single mean bar shows only the center and hides the spread — width, skew, and outliers — which is the entire question for a distribution. Use a histogram, box plot, violin, or ECDF instead. This is the "wrong chart for the data type" pitfall: a bar is for comparison, not for a distribution.

</details>

**Problem 3.** For each question, name the intent and the right chart: (a) "Do taller players score more points?" (b) "How did daily active users change over the last 90 days?" (c) "How does our budget split across 4 departments?"

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- (a) Two numeric columns, asking if they move together — that is a relationship. — _A relationship between two numerics is a scatter plot: each player is one point._
- (b) A numeric value over a time axis — that is a trend. — _A trend over time is a line chart; the connected line asserts the time order._
- (c) A total split into a few categorical parts — that is a composition. — _Part-to-whole with only 4 parts is fine as a stacked bar (or even a pie), but a plain bar per department is clearest._

**Answer:** (a) Relationship → scatter of height vs points. (b) Trend → line chart of daily active users over the 90 days. (c) Composition → stacked bar or a small bar chart per department (only 4 parts, so a pie would also work, but bars rank them more clearly).

</details>